# QC — FastQC + MultiQC

Запуск на сырые, trimmed или pr_trimmed FASTQ.
Параметр `label` определяет где искать файлы: `raw`, `trimmed`, `pr_trimmed`.

In [15]:
import os, sys
_CONDA_ENV = "/opt/conda/envs/bcr_env"
os.environ["PATH"] = _CONDA_ENV + "/bin:" + os.environ.get("PATH", "")
if os.path.isdir(_CONDA_ENV + "/lib/python3.11/site-packages"):
    sys.path.insert(0, _CONDA_ENV + "/lib/python3.11/site-packages")

In [16]:
import subprocess
from pathlib import Path

def log(msg):
    print(f"[qc] {msg}")

In [17]:
def run_qc(volume, dataset, label):
    vol = Path(volume)
    if label == "raw":
        sd = vol / "raw" / dataset
    elif label == "trimmed":
        sd = vol / "results" / dataset / "trimmed" / "fastq"
    else:
        sd = vol / "results" / dataset / "pr_trimmed" / "fastq"

    if not sd.is_dir():
        log(f"ERROR: {sd} not found")
        return

    base = vol / "results" / dataset / f"qc_{label}"
    fqo = base / "fastqc"
    mqo = base / "multiqc"
    fqo.mkdir(parents=True, exist_ok=True)
    mqo.mkdir(parents=True, exist_ok=True)

    ffs = sorted(sd.glob("*.fastq.gz"))
    log(f"=== QC {dataset} ({label}): {len(ffs)} files ===")

    subprocess.run(["fastqc","-t","4","-q","--noextract"] +
                   [str(f) for f in ffs] + ["-o", str(fqo)])
    log(f"  FastQC done")

    subprocess.run(["multiqc", str(fqo), "-o", str(mqo),
                    "-n", f"{dataset}_{label}_multiqc", "-f"])
    log(f"  MultiQC done")
    log(f"=== QC {dataset} ({label}) COMPLETE ===")

### Примеры запуска

PRJEB40348, PRJNA848968, PRJNA1247978, PRJNA900592.

In [14]:
# QC сырых
run_qc("/data/user/epishkin", "PRJEB40348", "raw")

[qc] === QC PRJEB40348 (raw): 70 files ===
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
appli


/// ]8;id=15259421;https://multiqc.info\MultiQC]8;;\ 🔍 v1.35

       file_search | Search path: /data/user/epishkin/results/PRJEB40348/qc_raw/fastqc
         searching | ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% 140/140  0m  
            fastqc | Found 70 reports
     write_results | Data        : /data/user/epishkin/results/PRJEB40348/qc_raw/multiqc/PRJEB40348_raw_multiqc_data
     write_results | Report      : /data/user/epishkin/results/PRJEB40348/qc_raw/multiqc/PRJEB40348_raw_multiqc.html
           multiqc | MultiQC complete


[qc]   MultiQC done
[qc] === QC PRJEB40348 (raw) COMPLETE ===


In [20]:
# QC после adapter_trim (все 4 датасета)
for _ds in ["PRJEB40348", "PRJNA848968", "PRJNA1247978", "PRJNA900592"]:
    run_qc("/data/user/epishkin", _ds, "trimmed")

[qc] === QC PRJEB40348 (trimmed): 70 files ===
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
a


/// ]8;id=11255761;https://multiqc.info\MultiQC]8;;\ 🔍 v1.35

       file_search | Search path: /data/user/epishkin/results/PRJEB40348/qc_trimmed/fastqc
         searching | ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% 140/140  0m  
            fastqc | Found 70 reports
     write_results | Data        : /data/user/epishkin/results/PRJEB40348/qc_trimmed/multiqc/PRJEB40348_trimmed_multiqc_data   (overwritten)
     write_results | Report      : /data/user/epishkin/results/PRJEB40348/qc_trimmed/multiqc/PRJEB40348_trimmed_multiqc.html   (overwritten)
           multiqc | MultiQC complete


[qc]   MultiQC done
[qc] === QC PRJEB40348 (trimmed) COMPLETE ===


In [ ]:
# QC после primer_trim
run_qc("/data/user/epishkin", "PRJEB40348", "pr_trimmed")